***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [9. 实践部分](9_1_visualisation-inspection.ipynb)
    * 上一节： [9.1 数据检查、可视化与初步质量控制](9_1_visualisation-inspection.ipynb)
    * 下一节： [9.3 连续谱基础成像](9_3_continuum_imaging.ipynb)

***


## 9.2 基础校准流程：从校准源到目标场

基础连续谱校准把第 8 章的测量方程落实为一条处理链。带通校准源约束频率响应，相位校准源跟踪时间变化的复增益，通量校准源固定绝对振幅标尺，最后这些解被转移到目标场。每一步都在求解天线基 Jones 项，只是求解轴和物理含义不同。

实践中常把带通和时间增益分开处理。带通主要随频率变化，通常要求亮而谱形平滑的校准源；时间增益主要随观测时刻变化，通常依赖靠近目标场、反复插入的相位校准源。把两者混成一个自由度过高的问题，会增加噪声吸收和解退化风险。


### 9.2.1 先解带通，再解时间增益

若带通项记为 $B_p(\nu)$，时间增益记为 $G_p(t)$，目标场观测近似为

$$
V_{pq}^{\rm obs}(t,\nu)=G_p(t)B_p(\nu)X_{pq}(t,\nu)B_q^*(\nu)G_q^*(t)+N_{pq}.
$$

在高信噪比带通源上，可以逐通道求解 $B_p(\nu)$；在相位校准源上，可以对频率平均后的数据求解 $G_p(t)$。这种分步处理隐含了一个物理假设：频率响应和时间响应在所选解间隔内可以近似分离。若接收机响应快速随时间变化，或频率方向存在强 RFI，这个假设就需要重新检查。


![带通与时间增益的分步求解](figures/practical_calibration_bandpass_gain.png)

**图 9.2.1** 简化带通源和相位校准源上的求解结果。左侧为天线 4 的带通振幅和相位，右侧为天线 2 的时间增益。两类解都是天线基复增益，但样本轴分别是频率和时间。


### 9.2.2 解转移到目标场

`applycal` 的数学含义是把求得的 Jones 链从目标数据中除去。若目标数据的真实误差确实由 $G_p(t)B_p(\nu)$ 主导，那么校正后的可见度应回到目标模型附近，残差谱应显著下降。若校正后残差仍随通道、时间或基线呈现结构，则说明带通、时间增益、flag 或目标模型仍存在问题。


![带通和时间增益应用到目标场后的残差变化](figures/practical_calibration_applycal_target.png)

**图 9.2.2** 校准解应用到目标场后的效果。左图显示一条目标基线在校准前后的振幅，右图显示残差谱。校准后残差接近噪声水平，说明主要方向无关误差已被移除。


### 9.2.3 工作流判断

真实连续谱校准流程常包含 `setjy`、`bandpass`、`gaincal`、`fluxscale`、`applycal` 和校准后复查。命令名本身不是重点；关键是理解每一步约束的物理量和失败症状。带通失败会留下通道相关残差，时间增益失败会留下扫描或基线相关残差，通量标尺错误会系统性改变目标源亮度，flag 不足会让 RFI 污染解，flag 过度则会损失灵敏度和 $uv$ 覆盖。

完成基础校准后，才适合进入连续谱成像。否则成像器面对的不是天空 Fourier 采样问题，而是仍带有明显仪器项的可见度集。
